# 01 — CIFAR-100 Data Exploration
Class distribution, sample grid, and augmentation preview.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Install deps in Colab
# !pip install torch torchvision timm torch-geometric matplotlib seaborn scikit-learn tqdm


In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

DATA_ROOT = '../data'
raw_ds = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True,
                             transform=transforms.ToTensor())
class_names = raw_ds.classes
print(f'Classes: {len(class_names)}  |  Train samples: {len(raw_ds)}')

In [ ]:
# Class distribution bar chart
label_counts = Counter(raw_ds.targets)
counts = [label_counts[i] for i in range(100)]

fig, ax = plt.subplots(figsize=(20, 4))
ax.bar(range(100), counts, color='steelblue', width=0.8)
ax.set_xlabel('Class ID')
ax.set_ylabel('Sample Count')
ax.set_title('CIFAR-100 Class Distribution (Train)')
ax.set_xticks(range(0, 100, 5))
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=120)
plt.show()
print(f'Min: {min(counts)}  Max: {max(counts)}  (perfectly balanced)')

In [ ]:
# 10x10 sample grid (one per class)
fig, axes = plt.subplots(10, 10, figsize=(14, 14))
shown = set()
idx = 0
samples = {}
for img, label in raw_ds:
    if label not in shown:
        samples[label] = img
        shown.add(label)
    if len(shown) == 100:
        break

for class_id, ax in enumerate(axes.flat):
    img = samples[class_id].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[class_id], fontsize=5)
    ax.axis('off')

plt.suptitle('One Sample per CIFAR-100 Class', fontsize=14)
plt.tight_layout()
plt.savefig('../results/sample_grid.png', dpi=150)
plt.show()

In [ ]:
# Augmentation preview: original vs augmented side-by-side
from src.preprocess import build_train_transform, build_eval_transform

aug_transform  = build_train_transform(input_size=128)
eval_transform = build_eval_transform(input_size=128)

raw_img, _ = raw_ds[0]  # tensor (3, 32, 32)
from PIL import Image
pil_img = transforms.ToPILImage()(raw_img)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i in range(6):
    orig = eval_transform(pil_img).permute(1, 2, 0).numpy()
    aug  = aug_transform(pil_img).permute(1, 2, 0).numpy()
    orig = np.clip(orig * 0.5 + 0.5, 0, 1)  # un-normalize roughly
    aug  = np.clip(aug  * 0.5 + 0.5, 0, 1)
    axes[0, i].imshow(orig)
    axes[0, i].axis('off')
    axes[1, i].imshow(aug)
    axes[1, i].axis('off')

axes[0, 0].set_title('Original', fontsize=10)
axes[1, 0].set_title('Augmented', fontsize=10)
plt.suptitle('Data Augmentation Preview', fontsize=13)
plt.tight_layout()
plt.savefig('../results/augmentation_preview.png', dpi=120)
plt.show()